# Figure 4: ML-predicted scaling relations and CHE volcano

This notebook downloads the four Figure 4 input files from the `figure4/` folder of the `doublelayer/EActa_2026_AI-acceleration` GitHub repository on each run. It verifies the supplied training and validation cohorts, applies the registered thermochemical corrections once, and generates the two-panel figure from the prediction cohort. In Colab, the notebook fetches its inputs directly; in a local checkout, run it from `figure4_outputs/`.


In [1]:
##########################
##-Installing libraries-##
##########################

import sys
if 'google.colab' in sys.modules:
    !pip install -q numpy pandas matplotlib plotly

In [2]:
"""Build Figure 4 v23 from the prediction table supplied for the model screen.

The plotting design is preserved from the student's approved Figure 4 v17.
The training/validation split and prediction-cohort labels are audited from
the supplied role files before the publication figure is produced.
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import urllib.request
from pathlib import Path

HERE = Path('/content') if Path('/content').is_dir() else Path.cwd()
OUTPUT_DIR = HERE / 'inputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib_cache"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


GITHUB_FIGURE4_BASE = (
    "https://raw.githubusercontent.com/doublelayer/"
    "EActa_2026_AI-acceleration/main/figure4"
)
SOURCE_DIR = HERE / 'source_data'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
INPUT_NAMES = (
    "predictions.csv",
    "structure_model_roles.csv",
    "model_structure_descriptors.csv",
    "split_config.json",
)
for input_name in INPUT_NAMES:
    input_path = SOURCE_DIR / input_name
    temporary_path = input_path.with_suffix(input_path.suffix + '.download')
    temporary_path.unlink(missing_ok=True)
    urllib.request.urlretrieve(f"{GITHUB_FIGURE4_BASE}/{input_name}", temporary_path)
    if not temporary_path.read_bytes().strip():
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Downloaded input is empty: {input_name}")
    temporary_path.replace(input_path)
empty_inputs = [
    name for name in INPUT_NAMES
    if not (SOURCE_DIR / name).read_bytes().strip()
]
if empty_inputs:
    raise RuntimeError(
        "The GitHub Figure 4 input file(s) are empty: "
        + ", ".join(empty_inputs)
        + ". Populate these files in the repository before running the figure notebook."
    )
PREDICTIONS_FILE = SOURCE_DIR / "predictions.csv"
ROLES_FILE = SOURCE_DIR / "structure_model_roles.csv"
DESCRIPTORS_FILE = SOURCE_DIR / "model_structure_descriptors.csv"
SPLIT_FILE = SOURCE_DIR / "split_config.json"
BASE = OUTPUT_DIR / "Figure_4_predicted_scaling_volcano_v23"
FIGURE_VERSION = 23
COHORT_AUDIT = OUTPUT_DIR / "Figure_4_cohort_audit_v23.json"
FIT_STATISTICS = OUTPUT_DIR / "Figure_4_scaling_fit_statistics_v23.csv"

WIDTH_CM, HEIGHT_CM, FONT_PT = 17.0, 8.5, 8
METALS = ["Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
JMOL_COLORS = {
    "Cr": "#8A99C7", "Mn": "#9C7AC7", "Fe": "#E06633",
    "Co": "#F090A0", "Ni": "#50D050", "Cu": "#C88033",
}
OOH_MPL_MARKER, O_MPL_MARKER, VOLCANO_MPL_MARKER = "s", "o", "D"
OOH_PLOTLY_MARKER, O_PLOTLY_MARKER, VOLCANO_PLOTLY_MARKER = "square", "circle", "diamond"

# Dense-screen markers.  The volcano is deliberately lighter because it
# contains more than two thousand overlapping predictions in one panel.
SCALING_MARKER_AREA = 10
VOLCANO_MARKER_AREA = 9
SCALING_FACE_ALPHA = 0.34
SCALING_EDGE_ALPHA = 0.26
VOLCANO_FACE_ALPHA = 0.18
VOLCANO_EDGE_ALPHA = 0.14
MARKER_EDGE_WIDTH = 0.30


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def audit_supplied_cohorts() -> dict:
    """Verify the structure roles and the contributor-defined split."""
    roles = pd.read_csv(ROLES_FILE)
    descriptors = pd.read_csv(DESCRIPTORS_FILE)
    split = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
    role_columns = ["O_model_role", "OH_model_role", "OOH_model_role"]

    if roles["name"].duplicated().any():
        raise ValueError("Duplicate catalyst names were found in the structure-role table.")
    if roles[role_columns].nunique(axis=1).ne(1).any():
        raise ValueError("O, OH, and OOH roles differ for at least one structure.")
    role = roles[role_columns[0]]
    role_counts = role.value_counts().to_dict()
    expected_roles = {"training": 50, "validation": 10, "prediction": 1288}
    if role_counts != expected_roles:
        raise ValueError(f"Unexpected structure-role counts: {role_counts}")

    descriptor_keys = ["name", "ads_site"]
    if descriptors.duplicated(descriptor_keys).any():
        raise ValueError("Duplicate catalyst and adsorption-center descriptor rows were found.")
    descriptor_role_counts = descriptors["O_model_role"].value_counts().to_dict()
    expected_descriptor_roles = {"prediction": 2576, "training": 100, "validation": 20}
    if descriptor_role_counts != expected_descriptor_roles:
        raise ValueError(f"Unexpected descriptor role counts: {descriptor_role_counts}")

    expected_split = {
        "training_structures": 50,
        "training_site_rows": 70,
        "validation_max_structures": 10,
        "validation_site_rows": 13,
    }
    observed_split = {key: int(split[key]) for key in expected_split}
    if observed_split != expected_split:
        raise ValueError(f"Unexpected training/validation split: {observed_split}")

    prediction_names = set(roles.loc[role.eq("prediction"), "name"])
    return {
        "input_source": GITHUB_FIGURE4_BASE,
        "input_sha256": {
            path.name: sha256(path)
            for path in (PREDICTIONS_FILE, ROLES_FILE, DESCRIPTORS_FILE, SPLIT_FILE)
        },
        "structure_role_counts": role_counts,
        "descriptor_site_role_counts": descriptor_role_counts,
        "model_split": {
            "training_structures": 50,
            "training_points": 70,
            "validation_structures": 10,
            "validation_points": 13,
        },
        "prediction_names": prediction_names,
    }




In [3]:
def build_analysis_table(cohort_audit: dict) -> pd.DataFrame:
    predictions = pd.read_csv(PREDICTIONS_FILE)
    keys = ["name", "ads_site"]
    required = set(keys + ["target", "predicted_eV"])
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Predictions are missing columns: {sorted(missing)}")
    if predictions.duplicated(keys + ["target"]).any():
        raise ValueError("Duplicate catalyst, adsorption-center, and target rows were found.")
    if set(predictions["target"]) != {"G_OH", "G_O", "G_OOH"}:
        raise ValueError("The prediction table must contain G_OH, G_O, and G_OOH.")
    predictions["predicted_eV"] = pd.to_numeric(predictions["predicted_eV"], errors="coerce")
    if not np.isfinite(predictions["predicted_eV"]).all():
        raise ValueError("The prediction table contains a missing or non-finite value.")

    frame = predictions.pivot(index=keys, columns="target", values="predicted_eV").reset_index()
    if frame[["G_OH", "G_O", "G_OOH"]].isna().any().any():
        raise ValueError("Every catalyst and adsorption center must have all three predictions.")
    # Plot complete combinations actually supplied in the current CSV.
    # Some structures have one represented center; retain its identity.
    if not frame["ads_site"].isin(["metal1", "metal2"]).all():
        raise ValueError("An unknown adsorption-center label was supplied.")
    descriptors = pd.read_csv(DESCRIPTORS_FILE)
    descriptor_keys = set(map(tuple, descriptors[keys].to_numpy()))
    if not set(map(tuple, frame[keys].to_numpy())) <= descriptor_keys:
        raise ValueError("A prediction has no matching structure and adsorption-center descriptor.")
    if set(frame["name"]) != cohort_audit["prediction_names"]:
        raise ValueError("Prediction-table names differ from the supplied prediction-role labels.")

    parts = frame["name"].str.split("_", expand=True)
    if parts.shape[1] < 7:
        raise ValueError("A catalyst name cannot be parsed into the declared molecular components.")
    frame["cohort"] = "DASC_Bare_Catalysts_RPBE_D4"
    frame["metal1"], frame["metal2"] = parts[1], parts[2]
    frame["dialdehyde"], frame["diamine"] = parts[3], parts[4]
    frame["bridge"] = parts[5].str[-2:]
    frame["metal"] = np.where(frame["ads_site"].eq("metal1"), frame["metal1"], frame["metal2"])
    frame["other_metal"] = np.where(frame["ads_site"].eq("metal1"), frame["metal2"], frame["metal1"])

    per_catalyst = frame.groupby("name", sort=False).size()
    if not per_catalyst.between(1, 2).all():
        raise ValueError("Each structure must have one or two supplied adsorption centers.")

    frame["retained"] = frame["G_OH"].ge(0.00) & frame["G_O"].le(4.92)
    steps = np.column_stack([
        4.92 - frame["G_OOH"],
        frame["G_OOH"] - frame["G_O"],
        frame["G_O"] - frame["G_OH"],
        frame["G_OH"],
    ])
    frame["U_L"] = steps.min(axis=1)
    frame["overpotential"] = 1.23 - frame["U_L"]
    labels = np.array(["O2 to OOH", "OOH to O", "O to OH", "OH to water"])
    frame["potential_determining_step"] = labels[steps.argmin(axis=1)]

    # Source targets named G_* contain adsorption potential energies, as
    # clarified by the author. Preserve them and apply corrections once.
    net_corrections = {'OH': 0.05, 'O': 0.05, 'OOH': 0.10}
    for species, correction in net_corrections.items():
        frame[f'E_ads_{species}'] = frame[f'G_{species}']
        frame[f'G_{species}'] = frame[f'E_ads_{species}'] + correction
        assert np.allclose(
            frame[f'G_{species}'] - frame[f'E_ads_{species}'],
            correction, rtol=0, atol=1e-12,
        )
    # Preserve the original electronic-energy filters.
    steps = np.column_stack([
        4.92 - frame['G_OOH'], frame['G_OOH'] - frame['G_O'],
        frame['G_O'] - frame['G_OH'], frame['G_OH'],
    ])
    frame['U_L'] = steps.min(axis=1)
    frame['overpotential'] = 1.23 - frame['U_L']
    frame['potential_determining_step'] = labels[steps.argmin(axis=1)]
    return frame




In [4]:
def main() -> None:
    outputs = [BASE.with_suffix(ext) for ext in (".csv", ".png", ".svg", ".pdf", ".html")]
    outputs.extend([COHORT_AUDIT, FIT_STATISTICS])
    collisions = [path for path in outputs if path.exists()]
    if collisions:
        raise FileExistsError(f"Figure 4 v{FIGURE_VERSION} output exists: {collisions[0]}")

    cohort_audit = audit_supplied_cohorts()
    frame = build_analysis_table(cohort_audit)
    frame.to_csv(BASE.with_suffix(".csv"), index=False)
    data = frame.loc[frame["retained"]].copy().reset_index(drop=True)
    if len(data) < 2:
        raise ValueError("Fewer than two complete combinations pass the declared energy filters.")

    descriptor_min, descriptor_max = float(data["G_OH"].min()), float(data["G_OH"].max())
    descriptor_pad = 0.08 * (descriptor_max - descriptor_min)
    descriptor_limits = (descriptor_min - descriptor_pad, descriptor_max + descriptor_pad)
    x_line = np.linspace(*descriptor_limits, 501)
    fit_o = np.polyfit(data["G_OH"], data["G_O"], 1)
    fit_ooh = np.polyfit(data["G_OH"], data["G_OOH"], 1)
    pd.DataFrame([
        {"relation": "O versus OH", "slope": fit_o[0], "intercept_eV": fit_o[1]},
        {"relation": "OOH versus OH", "slope": fit_ooh[0], "intercept_eV": fit_ooh[1]},
    ]).to_csv(FIT_STATISTICS, index=False)
    y_o = np.polyval(fit_o, x_line)
    y_ooh = np.polyval(fit_ooh, x_line)
    fit_steps = np.column_stack([4.92-y_ooh, y_ooh-y_o, y_o-x_line, x_line])
    eta_fit = 1.23 - fit_steps.min(axis=1)
    ideal_eta = 1.23 - np.minimum(x_line, 4.92-(x_line+2.46))
    eta_tick_max = math.ceil(max(float(data["overpotential"].max()), float(eta_fit.max()),
                                 float(ideal_eta.max())) / 0.2) * 0.2
    y_min = min(float(data["G_O"].min()), float(data["G_OOH"].min()),
                float(y_o.min()), float(y_ooh.min()))
    y_max = max(float(data["G_O"].max()), float(data["G_OOH"].max()),
                float(y_o.max()), float(y_ooh.max()))
    y_pad = 0.06 * (y_max-y_min)
    y_limits = (y_min-y_pad, y_max+y_pad)

    hover_columns = [
        "name", "ads_site", "metal", "other_metal", "dialdehyde", "diamine", "bridge",
        "G_OH", "G_O", "G_OOH", "U_L", "overpotential", "potential_determining_step",
    ]
    hover = (
        "<b>%{customdata[0]}</b><br>Adsorption-center label: %{customdata[1]} (%{customdata[2]})"
        "<br>Other metal: %{customdata[3]}<br>Dialdehyde: %{customdata[4]}"
        "<br>Diamine: %{customdata[5]}<br>Bridge: %{customdata[6]}"
        "<br>Predicted OH adsorption free energy: %{customdata[7]:.2f} eV"
        "<br>Predicted O adsorption free energy: %{customdata[8]:.2f} eV"
        "<br>Predicted OOH adsorption free energy: %{customdata[9]:.2f} eV"
        "<br>Limiting potential: %{customdata[10]:.2f} V"
        "<br>Predicted ORR overpotential: %{customdata[11]:.2f} V"
        "<br>Potential-determining step: %{customdata[12]}<extra></extra>"
    )

    interactive = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
    for metal in METALS:
        group = data.loc[data["metal"].eq(metal)]
        if group.empty:
            continue
        custom = group[hover_columns].to_numpy()
        interactive.add_trace(go.Scatter(
            x=group["G_OH"], y=group["G_OOH"], mode="markers", name=f"{metal} OOH", zorder=2,
            showlegend=False, customdata=custom, hovertemplate="<b>OOH</b><br>"+hover,
            marker=dict(symbol=OOH_PLOTLY_MARKER, color=JMOL_COLORS[metal], size=5,
                        opacity=0.38, line=dict(color="black", width=0.35)),
        ), row=1, col=1)
        interactive.add_trace(go.Scatter(
            x=group["G_OH"], y=group["G_O"], mode="markers", name=f"{metal} O", zorder=2,
            showlegend=False, customdata=custom, hovertemplate="<b>O</b><br>"+hover,
            marker=dict(symbol=O_PLOTLY_MARKER, color=JMOL_COLORS[metal], size=5,
                        opacity=0.38, line=dict(color="black", width=0.35)),
        ), row=1, col=1)
        interactive.add_trace(go.Scatter(
            x=group["G_OH"], y=group["overpotential"], mode="markers", name=metal, zorder=2,
            showlegend=False, customdata=custom, hovertemplate="<b>CHE activity</b><br>"+hover,
            marker=dict(symbol=VOLCANO_PLOTLY_MARKER, color=JMOL_COLORS[metal], size=4,
                        opacity=0.22, line=dict(color="black", width=0.25)),
        ), row=1, col=2)
    for y, column, style in [
        (y_ooh, 1, dict(color="#333333", width=1.2)),
        (y_o,   1, dict(color="#555555", width=1.2, dash="dash")),
        (eta_fit, 2, dict(color="#333333", width=1.2)),
        (ideal_eta, 2, dict(color="#777777", width=0.9, dash="dot")),
    ]:
        interactive.add_trace(go.Scatter(
            x=x_line, y=y, mode="lines", showlegend=False, hoverinfo="skip", zorder=0, line=style,
        ), row=1, col=column)
    interactive.add_trace(go.Scatter(
        x=[1.23], y=[0.0], mode="markers", showlegend=False, zorder=3,
        marker=dict(symbol="star", size=10, color="#F2C14E", line=dict(color="black", width=0.55)),
        hovertemplate="OH adsorption free energy: 1.23 eV<br>Overpotential: 0.00 V<extra></extra>",
    ), row=1, col=2)
    interactive.update_layout(
        width=1250, height=560, paper_bgcolor="white", plot_bgcolor="white", showlegend=False,
        font=dict(family="Arial, sans-serif", size=8, color="black"),
        margin=dict(l=120, r=55, t=35, b=75), hovermode="closest",
    )
    interactive.update_xaxes(title_text="OH adsorption free energy / eV", range=list(descriptor_limits),
        tickmode="array", tickvals=[0, 1, 2], ticktext=["0", "1", "2"],
        showline=True, linewidth=1, linecolor="black", mirror=True, row=1, col=1)

    interactive.update_xaxes(title_text="OH adsorption free energy / eV", range=list(descriptor_limits),
        tickmode="array", tickvals=[0, 1, 2], ticktext=["0", "1", "2"],
        showline=True, linewidth=1, linecolor="black", mirror=True, row=1, col=2)

    interactive.update_yaxes(title_text="O and OOH<br>adsorption free energy / eV",
        range=list(y_limits), tickmode="array", tickvals=[1, 2, 3, 4, 5], ticktext=["1", "2", "3", "4", "5"], showline=True, linewidth=1, linecolor="black", mirror=True, row=1, col=1)
    interactive.update_yaxes(title_text="Predicted ORR overpotential / V", range=[eta_tick_max, -0.14],
        tickmode="array", tickvals=[0, 1, 2], ticktext=["0", "1", "2"],
        showline=True, linewidth=1, linecolor="black", mirror=True, row=1, col=2)

    for xref, yref, label in [("x domain", "y domain", "(a)"), ("x2 domain", "y2 domain", "(b)")]:
        interactive.add_annotation(x=-0.11, y=1.04, xref=xref, yref=yref, text=label,
            showarrow=False, font=dict(size=8), xanchor="right", yanchor="bottom")
    interactive.write_html(BASE.with_suffix(".html"), include_plotlyjs=True)

    plt.rcParams.update({"font.size": FONT_PT, "font.family": "sans-serif"})
    figure, (scaling_ax, volcano_ax) = plt.subplots(
        1, 2, figsize=(WIDTH_CM/2.54, HEIGHT_CM/2.54)
    )
    figure.subplots_adjust(left=0.13, right=0.98, top=0.88, bottom=0.16, wspace=0.42)
    for metal in METALS:
        group = data.loc[data["metal"].eq(metal)]
        if group.empty:
            continue
        scaling_face = [to_rgba(JMOL_COLORS[metal], SCALING_FACE_ALPHA)]
        scaling_edge = [to_rgba("black", SCALING_EDGE_ALPHA)]
        volcano_face = [to_rgba(JMOL_COLORS[metal], VOLCANO_FACE_ALPHA)]
        volcano_edge = [to_rgba("black", VOLCANO_EDGE_ALPHA)]
        scaling_ax.scatter(
            group["G_OH"], group["G_OOH"], marker=OOH_MPL_MARKER, s=SCALING_MARKER_AREA,
            facecolors=scaling_face, edgecolors=scaling_edge, linewidths=MARKER_EDGE_WIDTH, zorder=3,
        )
        scaling_ax.scatter(
            group["G_OH"], group["G_O"], marker=O_MPL_MARKER, s=SCALING_MARKER_AREA,
            facecolors=scaling_face, edgecolors=scaling_edge, linewidths=MARKER_EDGE_WIDTH, zorder=3,
        )
        volcano_ax.scatter(
            group["G_OH"], group["overpotential"], marker=VOLCANO_MPL_MARKER,
            s=VOLCANO_MARKER_AREA, facecolors=volcano_face, edgecolors=volcano_edge,
            linewidths=MARKER_EDGE_WIDTH, zorder=3,
        )
    scaling_ax.plot(x_line, y_ooh, color="#333333", linewidth=1.0, zorder=1)
    scaling_ax.plot(x_line, y_o, color="#555555", linewidth=1.0, linestyle="--", zorder=1)
    volcano_ax.plot(x_line, eta_fit, color="#333333", linewidth=1.0, zorder=1)
    volcano_ax.plot(x_line, ideal_eta, color="#777777", linewidth=0.8, linestyle=":", zorder=1)
    volcano_ax.scatter([1.23], [0.0], marker="*", s=66, facecolor="#F2C14E",
        edgecolor="black", linewidths=0.45, zorder=5)
    scaling_ax.set_xlabel("OH adsorption free energy / eV")
    scaling_ax.set_ylabel("O and OOH\nadsorption free energy / eV")
    volcano_ax.set_xlabel("OH adsorption free energy / eV")
    volcano_ax.set_ylabel("Predicted ORR overpotential / V")
    scaling_ax.set_xticks([0, 1, 2])
    scaling_ax.set_yticks([1, 2, 3, 4, 5])
    volcano_ax.set_xticks([0, 1, 2])
    volcano_ax.set_yticks([0, 1, 2])
    volcano_ax.set_ylim(eta_tick_max, -0.14)
    scaling_ax.set_ylim(*y_limits)
    for axis, panel in ((scaling_ax, "(a)"), (volcano_ax, "(b)")):
        axis.set_xlim(*descriptor_limits)
        axis.set_box_aspect(1)
        axis.tick_params(direction="out", width=0.8, length=3, labelsize=8, colors="black")
        axis.text(-0.11, 1.035, panel, transform=axis.transAxes, clip_on=False,
            ha="left", va="bottom", fontsize=8, fontweight="bold")
        axis.set_facecolor("white")
        for spine in axis.spines.values():
            spine.set_color("black")
            spine.set_linewidth(0.8)
    figure.patch.set_facecolor("white")
    figure.savefig(BASE.with_suffix(".png"), dpi=600, facecolor="white", transparent=False)
    figure.savefig(BASE.with_suffix(".svg"), format="svg", facecolor="white", transparent=False)
    figure.savefig(BASE.with_suffix(".pdf"), format="pdf", facecolor="white", transparent=False)
    plt.close(figure)

    serializable_audit = {key: value for key, value in cohort_audit.items() if key != "prediction_names"}
    serializable_audit.update({
        "prediction_structures_in_figure_input": int(frame["name"].nunique()),
        "prediction_points_in_figure_input": int(len(frame)),
        "retained_structures_after_energy_filters": int(data["name"].nunique()),
        "retained_points_after_energy_filters": int(len(data)),
        "figure_version": FIGURE_VERSION,
        "visual_source": "student Figure 4 v17",
    })
    COHORT_AUDIT.write_text(
        json.dumps(serializable_audit, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    best = data.sort_values(["overpotential", "name", "ads_site"], kind="stable").head(3)
    print(f"Prediction source SHA-256: {sha256(PREDICTIONS_FILE)}")
    print(f"Rebuilt {len(frame)} combinations across {frame['name'].nunique()} catalysts.")
    print(f"Retained {len(data)} combinations across {data['name'].nunique()} catalysts.")
    print("Model split: 50 training structures / 70 points; "
          "10 validation structures / 13 points.")
    print(f"Saved Figure 4 v{FIGURE_VERSION}: {BASE.with_suffix('.png')}")
    print(best[["name", "ads_site", "metal", "other_metal", "bridge", "diamine", "overpotential"]]
          .to_string(index=False))




In [5]:
if __name__ == "__main__":
    main()


Prediction source SHA-256: e945663a27e930257c2d915f527b0a241bf7c1a5be74d244c97aa2f16d4c9617
Rebuilt 2216 combinations across 1288 catalysts.
Retained 2197 combinations across 1278 catalysts.
Model split: 50 training structures / 70 points; 10 validation structures / 13 points.
Saved Figure 4 v23: /mnt/c/users/vi/desktop/code/2026_article_ml-orr/figure4_outputs/Figure_4_predicted_scaling_volcano_v23.png
                     name ads_site metal other_metal bridge diamine  overpotential
KMC_Fe_Co_A4_B2_NNNNSS_q2   metal1    Fe          Co     SS      B2       0.110591
KMC_Fe_Co_A6_B2_NNNNSS_q2   metal1    Fe          Co     SS      B2       0.112308
KMC_Fe_Co_A5_B2_NNNNSS_q2   metal1    Fe          Co     SS      B2       0.114527
